# Gentrex on the benchmark

This notebook walks through Gentrex's pipeline on one benchmark report.
Then it shows how well and how fast the pipeline did on all reports of a
run, as a whole, for each of its four commands, for each field and for each
kind of report.

It reads what the scripts in [`evaluation/`](../evaluation/README.md) write
to `results/`. Run one of them first, such as
`evaluation/local/run_pipeline.sh`, and then open the notebook from the
Gentrex folder:

```bash
uv run --group notebook jupyter lab evaluation_results/pipeline.ipynb
```

The notebook computes no scores itself. Every number comes from the
`scores.json` of a run, so it always agrees with the run's `scores.md`.
[Run the benchmark](../docs/development/run-benchmark.md) explains how the
scorer counts.

Some tables quote values from the reports. After a run on real reports,
clear the outputs before you save or share the notebook.

In [ ]:
# The notebook group provides matplotlib and pandas.
# pyright: reportMissingImports=false

import json
import pathlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import pypdfium2
from matplotlib import ticker

sys.path.insert(0, "../gentrex-bench/scorer")

import score  # noqa: E402

# The notebook runs in evaluation_results/, so results/ is one folder up.
RESULTS = pathlib.Path("../results")
runs = {
    path.parent.name: json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(RESULTS.glob("gentrex-bench/*/scores.json"))
}
print(f"{len(runs)} scored runs in {(RESULTS / 'gentrex-bench').resolve()}")

In [ ]:
SURFACE, BLUE, INK, MUTED, GRID = "#fcfcfb", "#2a78d6", "#0b0b0b", "#52514e", "#e5e4e0"
COMMANDS = ("transcribe", "select", "extract", "normalize")
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.titlelocation": "left",
        "axes.titlesize": 11,
        "axes.labelcolor": MUTED,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": INK,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
    }
)


def percent(number):
    return "n/a" if number is None else f"{number:.1%}"


def with_interval(number):
    """Formats a main number with its 95% interval."""
    if number["value"] is None:
        return "n/a"
    return (
        f"{percent(number['value'])}"
        f" ({percent(number['low'])} to {percent(number['high'])})"
    )


def copy_of(run):
    """Returns the copy of the reports that a run read, such as native or scan."""
    return pathlib.Path(run["pdfs"]).name


def share_axis(ax):
    ax.set_xlim(0, 1.12)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
    ax.xaxis.set_major_formatter(ticker.PercentFormatter(1))
    ax.grid(axis="y", visible=False)
    ax.invert_yaxis()


def bars(numbers, title, label, shares=False):
    """Draws numbers as horizontal bars in the given order, each with its value.

    Args:
        numbers: A pandas Series of the numbers, by their labels.
        title: The title of the plot.
        label: Formats a number for the end of its bar.
        shares: Whether the numbers are shares, on an axis from 0% to 100%.
    """
    _, ax = plt.subplots(figsize=(7.5, 0.3 * len(numbers) + 1))
    ax.barh(numbers.index, numbers, height=0.55, color=BLUE)
    longest = 1 if shares else max(numbers.max(), 1e-9)
    for position, number in enumerate(numbers):
        ax.text(
            number + 0.012 * longest, position, label(number), va="center", fontsize=9
        )
    if shares:
        share_axis(ax)
    else:
        ax.set_xlim(0, longest * 1.12)
        ax.grid(axis="y", visible=False)
        ax.invert_yaxis()
    ax.set_title(title)
    plt.show()


def intervals(numbers, title):
    """Draws shares with their 95% intervals, one row per label."""
    _, ax = plt.subplots(figsize=(7.5, 0.4 * len(numbers) + 1))
    for position, number in enumerate(numbers.values()):
        if number["value"] is None:
            continue
        ax.hlines(position, number["low"], number["high"], color=BLUE, linewidth=2)
        ax.plot(
            number["value"],
            position,
            "o",
            color=BLUE,
            markersize=9,
            markeredgecolor=SURFACE,
            markeredgewidth=2,
            clip_on=False,
            zorder=3,
        )
        ax.text(
            number["high"] + 0.015,
            position,
            percent(number["value"]),
            va="center",
            fontsize=9,
        )
    ax.set_yticks(range(len(numbers)), list(numbers))
    share_axis(ax)
    ax.set_ylim(len(numbers) - 0.4, -0.6)
    # Room on the left, so that a marker at 0% doesn't sit on the labels.
    ax.set_xlim(-0.03, 1.12)
    ax.set_title(title)
    plt.show()

## The runs

One row per scored run, named by its folder under `results/gentrex-bench/`.
`Copy` is the copy of the reports that the run read: the native PDFs, or the
scanned, faxed, mixed or image copies. The intervals are 95% intervals. They
say how far a number could be off because the benchmark has only so many
reports.

In [ ]:
overview = pd.DataFrame(
    [
        {
            "Copy": copy_of(s),
            "Split": s["split"],
            "Reports": s["reports"],
            "Commands": ", ".join(c for c in COMMANDS if c in s["runs"]),
            "Extract model": s["runs"]["extract"]["settings"]["model"],
            "Report accuracy": with_interval(s["main"]["report_accuracy"]),
            "Variant F1": with_interval(s["main"]["variant_f1"]),
            "GPU-hours per 1,000 PDFs": s["performance"]["gpu_hours_per_1000_files"],
        }
        for s in runs.values()
    ],
    index=list(runs),
)
overview

Everything below is about one run, `RUN`, and the walkthrough follows one
of its reports, `REPORT`. By default these are the newest run and its first
report. To see others, put a folder name from the table above and a report
name such as `"SYN-0004"` in their place.

In [ ]:
RUN = list(runs)[-1]
scores = runs[RUN]
REPORT = sorted(scores["by_report"])[0]


def run_folder(command):
    """Returns the run folder of a command of RUN, or None if the run skipped it."""
    record = scores["runs"].get(command)
    if record is None:
        return None
    folder = pathlib.Path(record["path"])
    # Scores copied from another machine still name that machine's folders.
    return folder if folder.exists() else RESULTS / command / folder.name


def first_existing(*paths):
    """Returns the first of the paths that is on this machine, or None."""
    return next((path for path in paths if path.exists()), None)


# The benchmark version, for a run that was scored on another machine.
VERSION = (
    pathlib.Path("../data/gentrex-bench") / pathlib.Path(scores["truth"]).parent.name
)


def records(command, name):
    """Returns the records of REPORT in a file of a command's run folder."""
    folder = run_folder(command)
    if folder is None:
        print(f"{RUN} skipped gentrex {command}")
        return []
    if not (folder / name).is_file():
        print(f"{folder / name} is not on this machine")
        return []
    lines = (folder / name).read_text(encoding="utf-8").splitlines()
    found = [json.loads(line) for line in lines]
    return [record for record in found if record["file"] == f"{REPORT}.pdf"]


print(
    f"{RUN}: {scores['reports']} reports of the split {scores['split']},"
    f" {copy_of(scores)} copy; following {REPORT}"
)

## Walk through the pipeline

Gentrex turns the PDF of a genetic test report into one record. Four
commands do it, each reading what the one before wrote:

| Command | What it does | Model |
| --- | --- | --- |
| `gentrex transcribe` | Reads the text of every page. Scanned pages go through OCR. | LightOnOCR-3-4B, for scanned pages |
| `gentrex select` | Keeps the pages with the result or the patient's clinical features. | Gemma 4 26B-A4B |
| `gentrex extract` | Turns the kept pages into one record, with the values as printed. | Gemma 4 31B |
| `gentrex normalize` | Adds standard values: ISO dates, categories and IDs. | SapBERT and EmbeddingGemma 2, on the CPU |

![The four commands of Gentrex, from PDF pages to a record with standard values](../docs/images/gentrex-arch.svg)

### The report

A benchmark report is synthetic. Code picked every value that it prints, and
a language model wrote only the text around them, so its true content is
known. The truth file below names the template that the report was printed
with, its kind of test, its result and its challenge tags, which each make
the report harder in one known way.

In [ ]:
truth_path = first_existing(
    pathlib.Path(scores["truth"]) / f"{REPORT}.json",
    VERSION / "truth" / f"{REPORT}.json",
)
case = json.loads(truth_path.read_text(encoding="utf-8"))["case"] if truth_path else {}
pd.Series(case, name=REPORT, dtype=object).drop("writer", errors="ignore").to_frame()

The first page of the PDF that `gentrex transcribe` read:

In [ ]:
pdf_path = first_existing(
    pathlib.Path(scores["pdfs"]) / f"{REPORT}.pdf",
    VERSION / "pdf" / copy_of(scores) / f"{REPORT}.pdf",
)
if pdf_path is None:
    print(f"{REPORT}.pdf is not on this machine")
pypdfium2.PdfDocument(pdf_path)[0].render(scale=1).to_pil() if pdf_path else None

### `gentrex transcribe`

A page with a text layer is `native`, and its text is read from the PDF. A
page without one is `scanned`. It is drawn as an image and sent to the OCR
model, which writes its text. One line of `pages.jsonl` holds one page.

If text is lost here, no later command can get it back. The scan and fax
copies of the benchmark show how much OCR loses.

In [ ]:
pages = records("transcribe", "pages.jsonl")
pd.DataFrame(
    [
        {
            "page": page["page"],
            "source": page["source"],
            "characters": len(page["text"]),
            "error": page["error"],
        }
        for page in pages
    ]
)

In [ ]:
# The start of the first page's text.
print(pages[0]["text"][:700] if pages else "")

### `gentrex select`

Most pages of a report only describe the test, its methods and its limits.
`gentrex select` asks the model two yes-or-no questions about every page. It
asks whether the page states the result or lists variants, and whether it
describes the patient's own clinical features. It reads the probability of
"yes" from the model's first token. A page is kept when one of the two probabilities is at
least 0.5 (`select.keep_threshold`).

Fewer pages make the next request shorter and leave out text that could
mislead the model, such as a variant from a cited study. A page that is
wrongly dropped takes its values with it.

In [ ]:
pd.DataFrame(
    [
        {"page": d["page"], **d["p_yes"], "keep": d["keep"], "error": d["error"]}
        for d in records("select", "decisions.jsonl")
    ]
)

### `gentrex extract`

The kept pages of a report go to the model in one request, together with the
extraction schema, which describes every field. The model server lets the
model write only JSON that follows the schema. A field that the report
doesn't state comes back empty.

Fields ending in `_raw` hold the report's own wording, and the field after
each holds the closest of a fixed list of values, as `classification_raw`
and `classification` do. The table shows the fields of the report itself.

In [ ]:
extraction = (records("extract", "extractions.jsonl") or [{}])[0]
data = extraction.get("data") or {}
pd.Series(
    {k: v for k, v in data.items() if k not in ("variants", "patient_phenotypes")},
    name="value",
    dtype=object,
).to_frame()

The variants of the report, one row each:

In [ ]:
pd.DataFrame(data.get("variants") or []).T

The patient's clinical features. After them come the copied values that
Gentrex couldn't find in the text of the pages (`not_in_pages`). A value in
this list was made up or reworded by the model, so it is the first to check
against the PDF.

In [ ]:
print("not_in_pages:", extraction.get("not_in_pages"))
pd.DataFrame(data.get("patient_phenotypes") or [])

### `gentrex normalize`

Laboratories print the same thing in many ways. `gentrex normalize` keeps
the record as it is and adds standard values next to it, without a model
server:

- dates as YYYY-MM-DD, a category for the specimen and one for the result,
- the gene's approved HGNC symbol and ID, and the disease's MONDO ID,
- whether each c. and p. change follows the HGVS syntax, and the variant's
  ClinVar ID,
- the closest HPO term for each clinical feature.

The first table puts the printed values next to the standard ones.

In [ ]:
normalized = (records("normalize", "normalized.jsonl") or [{}])[0].get("normalized")
normalized = normalized or {}
pd.DataFrame(
    [
        {
            "field": field,
            "printed": data.get("overall_result_raw" if "result" in field else field),
            "normalized": normalized.get(field),
        }
        for field in (
            "date_of_birth",
            "collection_date",
            "received_date",
            "report_date",
            "specimen_type",
            "overall_result",
        )
    ]
).set_index("field")

What it adds to each variant:

In [ ]:
pd.DataFrame(normalized.get("variants") or []).T

The HPO term of each clinical feature, with whether the two embedding
models agreed on it. The table is empty after a run with `--no-hpo`.

In [ ]:
pd.DataFrame(
    [
        {
            "feature": feature["text"],
            "hpo_id": term["hpo_id"],
            "label": term["label"],
            "models_agree": term["models_agree"],
        }
        for feature, term in zip(
            data.get("patient_phenotypes") or [],
            normalized.get("patient_phenotypes") or [],
            strict=False,
        )
    ]
)

### The score of this report

The scorer compares the record with the truth file, value by value. A value
is correct, missed, wrong or extra. Each value that isn't correct is charged
to the first command that lost it: `gentrex transcribe` if its text is not
in the transcribed pages, `gentrex select` if it is on a page that wasn't
kept, `gentrex extract` if the record has it wrong, and `gentrex normalize`
if the record is right and the standard value is not.

The report counts as correct when nothing clinically relevant is wrong.
That covers the result category, the karyotype, and every variant with its
gene, DNA change, copy number, classification and zygosity.

In [ ]:
summary = scores["by_report"][REPORT]
print(
    f"{REPORT}: {'correct' if summary['correct'] else 'not correct'};"
    f" variants matched {summary['variants']['matched']},"
    f" missed {summary['variants']['missed']}, extra {summary['variants']['extra']}"
)
errors = pd.DataFrame(
    [
        json.loads(line)
        for line in (RESULTS / "gentrex-bench" / RUN / "errors.jsonl")
        .read_text(encoding="utf-8")
        .splitlines()
    ],
    columns=[
        "report",
        "field",
        "result",
        "truth",
        "found",
        "command",
        "cause",
        "flagged",
    ],
)
errors[errors["report"] == REPORT].drop(columns="report")

## The whole pipeline

The main numbers of `RUN`, over all its reports:

- **Report accuracy** is the share of reports that count as correct, as
  defined above.
- **MRN and date of birth both right** is the share of reports that would be
  filed under the right patient.
- **Variant F1** combines **variant recall**, the share of true variants
  that Gentrex found, with **variant precision**, the share of Gentrex's
  variants that are true.

In [ ]:
labels = {
    "report_accuracy": "Report accuracy",
    "patient_match": "MRN and date of birth both right",
    "variant_f1": "Variant F1",
    "variant_precision": "Variant precision",
    "variant_recall": "Variant recall",
}
intervals(
    {label: scores["main"][name] for name, label in labels.items()},
    f"Main numbers of {RUN} ({scores['reports']} reports), with 95% intervals",
)
print(f"{scores['without_record']} of {scores['reports']} reports have no record")

Variants and clinical features are matched one to one with the truth and
then counted as matched, missed or extra. Clinical features are counted by
their text, and after `gentrex normalize` by their HPO ID.

In [ ]:
pd.DataFrame(scores["entities"]).T.drop(columns=["missed_by", "extra_by"])

## Each command

### Where the errors come from

The values that each command lost, got wrong or added, over all reports. A
command high in this chart is the one to work on.

In [ ]:
charges = pd.DataFrame(scores["charges"]).T
bars(
    charges.sum(axis=1).rename(lambda command: f"gentrex {command}"),
    "Missed, wrong and extra values charged to each command",
    label=lambda number: f"{number:.0f}",
)
charges

### Time and tokens

Each command records how long it ran and how many requests and tokens it
exchanged with its model server. The time is that of the command alone,
without the minutes a server takes to load its model. `gpus` is the number
of GPUs that the command's model server ran on, from `--gpus`.

In [ ]:
performance = scores["performance"]
commands = pd.DataFrame(performance["commands"]).T
commands.insert(
    0,
    "model",
    [
        (scores["runs"][c]["settings"].get("ocr") or scores["runs"][c]["settings"]).get(
            "model", ""
        )
        for c in commands.index
    ],
)
seconds = commands["seconds"].astype(float)
bars(
    seconds.rename(lambda command: f"gentrex {command}"),
    f"Seconds of each command for {performance['files']} PDFs",
    label=lambda number: f"{number:,.0f} s",
)
print(
    f"{performance['seconds_per_1000_files']} seconds and"
    f" {performance['gpu_hours_per_1000_files']} GPU-hours per 1,000 PDFs"
)
commands

### What each command did

The pages that `gentrex transcribe` read and sent to OCR, and the pages that
`gentrex select` kept, over all reports of the run.

In [ ]:
def all_records(command, name):
    folder = run_folder(command)
    if folder is None or not (folder / name).is_file():
        return []
    lines = (folder / name).read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines]


all_pages = all_records("transcribe", "pages.jsonl")
decisions = all_records("select", "decisions.jsonl")
pd.Series(
    {
        "PDFs": len({page["file"] for page in all_pages}),
        "pages": len(all_pages),
        "pages sent to OCR": sum(page["source"] == "scanned" for page in all_pages),
        "pages without text after transcribe": sum(
            not page["text"] for page in all_pages
        ),
        "pages that gentrex select decided on": len(decisions),
        "pages kept": sum(decision["keep"] for decision in decisions),
    },
    name=RUN,
).to_frame()

`gentrex extract` names the copied values that it can't find in the pages
(`not_in_pages`). `precision` is the share of the named values that were
wrong or extra, and `recall` the share of all wrong or extra copied values
that were named. The table is empty for a run from before this check.

In [ ]:
pd.Series(scores.get("not_in_pages") or {}, name=RUN, dtype=object).to_frame()

The accuracy of the standard values that `gentrex normalize` adds. A
standard value can only be right if the value it comes from was extracted
right, so these numbers include the errors of the commands before.

In [ ]:
fields = pd.DataFrame(scores["fields"]).T
fields[[name.startswith("normalized.") for name in fields.index]]

## Each field

The 15 fields with the lowest accuracy, and then all fields. `strict` counts
the values that are also the same character for character. The accuracy
accepts another spelling of the same value, such as `c.5266dupC` for
`c.5266dup`.

In [ ]:
fields = fields.sort_values("accuracy")
bars(
    fields["accuracy"].astype(float).head(15),
    "Accuracy of the 15 weakest fields",
    label=lambda number: f"{number:.0%}",
    shares=True,
)
fields

## Each kind of report

Report accuracy by challenge tag, weakest first, and then for every group.
The groups are the template (`lab`), the kind of test, the result, the
number of variants, and whether the template is in the dev set. A group
with few reports has a wide interval, so read small differences with
care.

In [ ]:
tags = sorted(
    scores["groups"].get("tag", {}).items(),
    key=lambda tag: tag[1]["report_accuracy"],
)
if tags:
    bars(
        pd.Series(
            [numbers["report_accuracy"] for _, numbers in tags],
            index=[f"{label} ({numbers['reports']})" for label, numbers in tags],
        ),
        "Report accuracy by challenge tag (number of reports)",
        label=lambda number: f"{number:.0%}",
        shares=True,
    )
pd.DataFrame(
    [
        {"group": group, "label": label, **numbers}
        for group, labels in scores["groups"].items()
        for label, numbers in labels.items()
    ]
).set_index(["group", "label"])

## Compare runs

The report accuracy of every scored run. Runs of the same setup on different
copies show what scanning and faxing cost. Runs of different setups on the
same copy show what a model, a prompt or a command is worth.

In [ ]:
intervals(
    {f"{name}, {copy_of(s)}": s["main"]["report_accuracy"] for name, s in runs.items()},
    "Report accuracy of each run, with its 95% interval",
)

`score.difference` compares two runs on the reports that both scored. It
gives the second run's report accuracy minus the first's, with a 95%
interval. If the interval includes 0, the two runs can't be told apart on
these reports.

In [ ]:
FIRST, SECOND = list(runs)[0], list(runs)[-1]
gain = score.difference(runs[FIRST], runs[SECOND])
print(
    f"{SECOND} minus {FIRST} on {gain['reports']} shared reports:"
    f" {gain['value']:+.1%} ({gain['low']:+.1%} to {gain['high']:+.1%})"
)

Report accuracy against GPU-hours per 1,000 PDFs, for the runs that were
scored with `--gpus`. Up and to the left is better.

In [ ]:
priced = {n: s for n, s in runs.items() if s["performance"]["gpu_hours_per_1000_files"]}
if priced:
    _, ax = plt.subplots(figsize=(7.5, 4))
    for name, s in priced.items():
        cost = s["performance"]["gpu_hours_per_1000_files"]
        accuracy = s["main"]["report_accuracy"]["value"]
        ax.plot(
            cost,
            accuracy,
            "o",
            color=BLUE,
            markersize=9,
            markeredgecolor=SURFACE,
            markeredgewidth=2,
            clip_on=False,
            zorder=3,
        )
        ax.annotate(
            f"{name}, {copy_of(s)}",
            (cost, accuracy),
            xytext=(8, 0),
            textcoords="offset points",
            va="center",
            fontsize=9,
        )
    ax.set_xlim(
        0,
        max(s["performance"]["gpu_hours_per_1000_files"] for s in priced.values())
        * 1.5,
    )
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
    ax.set_xlabel("GPU-hours per 1,000 PDFs")
    ax.set_title("Report accuracy and cost of each run")
    plt.show()
else:
    print("No run was scored with --gpus.")

## The errors of the run

Every value of `RUN` that isn't right, counted by command and field, and
then the first 50 with the truth and what Gentrex wrote. Filter `errors` to
study one field or command.

In [ ]:
errors.groupby(["command", "field"], dropna=False).size().to_frame("values")

In [ ]:
errors.head(50)